In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
#
# transformers
#   Loads SmolLM2 and its tokenizer/chat template.
#
# datasets
#   Converts our Pandas dataframe into a Hugging Face Dataset.
#
# trl
#   Hugging Face's post-training library.
#   We specifically use SFTTrainer for supervised fine-tuning.
#
# accelerate
#   Handles device placement/training infrastructure.
#
# pandas
#   Reads our CSV.
#
# tqdm
#   Progress bars.
#
#
# IMPORTANT:
#
# We are intentionally NOT using LoRA/QLoRA for this first
# experiment.
#
# SmolLM2-135M is tiny enough that a T4 can FULL fine-tune it.
#
# That means:
#
#       ALL ~135M parameters
#              ↓
#          trainable
#
# This is useful for our experiment because we're asking:
#
# "How much can a tiny general model specialize when its
# entire capacity is allowed to adapt to one task?"
# ============================================================

!pip install -q -U \
    transformers \
    datasets \
    trl \
    accelerate \
    pandas \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 whi

In [ ]:
# ============================================================
# GENERATION-EVALUATION DEPENDENCIES
# ============================================================
#
# Training metrics such as:
#
#     eval_loss
#     token accuracy
#     entropy
#
# use TEACHER FORCING.
#
# They answer:
#
# "Given the correct previous tokens, how well can the model
#  predict the next target token?"
#
#
# But our production behavior is AUTOREGRESSIVE GENERATION:
#
# paragraph
#      ↓
# model.generate()
#      ↓
# actual bullet output
#
#
# Therefore at the end of every epoch we will additionally
# calculate:
#
# ROUGE-1
# ROUGE-2
# ROUGE-L
# BERTScore Precision
# BERTScore Recall
# BERTScore F1
# Bullet-format compliance
# Predicted bullet count
# Reference bullet count
# Compression ratio
#
# ============================================================

!pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 574.9 kB/s eta 0:00:00


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import time
import math

import torch
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from trl import (
    SFTTrainer,
    SFTConfig
)

from tqdm.auto import tqdm
import numpy as np

from transformers import TrainerCallback

from rouge_score import rouge_scorer

from bert_score import score as bert_score

In [ ]:
# ============================================================
# HARDWARE CHECK
# ============================================================
#
# We expect something like:
#
# CUDA available: True
# GPU: Tesla T4
# VRAM: ~15 GB
#
#
# T4 notes:
#
# - Excellent FP16 support
# - Does NOT have the same native BF16 capabilities as
#   newer Ampere/Hopper GPUs
#
# Therefore:
#
#       FP16 = YES
#       BF16 = NO
# ============================================================

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab choose "
        "Runtime → Change runtime type → T4 GPU."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
#
# MODEL
#
# We start from the INSTRUCTION-TUNED checkpoint rather than
# the base SmolLM2 checkpoint.
#
# Why?
#
# It already understands:
#
#   system
#   user
#   assistant
#
# roles and already has basic summarization/instruction
# following capability.
#
# We're specializing that capability further.
#
#
# MAX_LENGTH
#
# Start with 2048 tokens on the T4.
#
# This is NOT the model's theoretical maximum context.
#
# It's our TRAINING sequence limit.
#
# Longer sequences:
#
#   + preserve longer documents
#   - consume substantially more VRAM
#   - reduce training throughput
#
# Once the experiment works, we can separately test 4096.
# ============================================================

MODEL_ID = (
    "HuggingFaceTB/SmolLM2-135M-Instruct"
)

TRAIN_CSV = (
    "/content/output_with_bullet_points.csv"
)

OUTPUT_DIR = (
    "/content/smollm2-135m-bullet-sft"
)

FINAL_MODEL_DIR = (
    "/content/smollm2-135m-bullet-specialist"
)

MAX_LENGTH = 2048

SEED = 42

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
#
# Expected:
#
# text
# source
# example_id
# bullet_points
#
#
# MODEL INPUT:
#
#       text
#
#
# SUPERVISED TARGET:
#
#       bullet_points
#
#
# source/example_id are metadata.
# They are NOT shown to the model.
# ============================================================

df = pd.read_csv(
    TRAIN_CSV
)

print(
    "Rows:",
    len(df)
)

print(
    "Columns:",
    df.columns.tolist()
)

df.head()

Rows: 500
Columns: ['text', 'source', 'example_id', 'bullet_points']


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
# ============================================================
# DATASET VALIDATION
# ============================================================

assert "text" in df.columns, (
    "Missing 'text' column"
)

assert "bullet_points" in df.columns, (
    "Missing 'bullet_points' column"
)


print("Dataset schema OK")

print()

if "source" in df.columns:

    print("Source distribution:")

    print(
        df["source"]
        .value_counts()
    )

Dataset schema OK

Source distribution:
source
ag_news          125
cnn_dailymail    125
xsum             125
email            125
Name: count, dtype: int64


In [ ]:
# ============================================================
# MINIMAL CLEANING
# ============================================================
#
# We DON'T rewrite or normalize the language.
#
# We want the model to learn from natural English.
#
# Only:
#
# - remove NULL input/output
# - convert to string
# - trim whitespace
# - remove empty examples
# ============================================================

df = df[
    [
        "text",
        "bullet_points"
    ]
].copy()


df = df.dropna(
    subset=[
        "text",
        "bullet_points"
    ]
)


df["text"] = (
    df["text"]
    .astype(str)
    .str.strip()
)


df["bullet_points"] = (
    df["bullet_points"]
    .astype(str)
    .str.strip()
)


df = df[
    (df["text"] != "")
    &
    (df["bullet_points"] != "")
].reset_index(drop=True)


print(
    "Usable examples:",
    len(df)
)

Usable examples: 500


In [ ]:
# ============================================================
# MANUAL QUALITY CHECK
# ============================================================
#
# This matters more than people often realize.
#
# SFT is imitation learning.
#
# If the targets:
#
#   hallucinate
#   miss important points
#   over-summarize
#   generate fixed numbers of bullets
#
# then our student learns those behaviors.
# ============================================================

for i in range(
    min(3, len(df))
):

    print("=" * 100)

    print("\nINPUT:\n")

    print(
        df.iloc[i]["text"]
    )

    print("\nTARGET:\n")

    print(
        df.iloc[i]["bullet_points"]
    )

    print()


INPUT:

Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39;s foreign policy advisers said Sunday.

TARGET:

- Powell says the large North Korean explosion was not a nuclear event.  
- U.S. officials do not believe the blast was linked to North Korea’s suspected nuclear weapons program.  
- The statement was made by President Bush’s foreign policy advisers on Sunday.


INPUT:

Jerusalem (CNN) -- Two attacks carried out against Palestinians in recent days, one in Jerusalem and the other on the West Bank, have prompted some uncomfortable questions in Israel about racism toward Arabs. In one incident, Jamal Julani, a 17-year-old Palestinian from East Jerusalem, was beaten unconscious by a group of teenagers in West Jerusalem. He spent several days in the hospital recovering from his injuries. On the same day, August 16, a taxi carr

In [ ]:
# ============================================================
# TASK INSTRUCTION
# ============================================================
#
# This makes our training:
#
#       TASK-SPECIFIC INSTRUCTION SFT
#
#
# Every example teaches:
#
# SYSTEM:
#     What behavior should you perform?
#
# USER:
#     Here is the document.
#
# ASSISTANT:
#     Here are the correct bullets.
#
#
# IMPORTANT:
#
# We deliberately NEVER say:
#
#     "generate 5 bullets"
#
# because:
#
#     bullet count = function(information density)
#
# not a fixed output format.
# ============================================================

SYSTEM_PROMPT = """
You convert English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, and trivial details.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add information unsupported by the source.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".
""".strip()

In [ ]:
# ============================================================
# TOKENIZER
# ============================================================
#
# SmolLM2-Instruct already has a chat template.
#
# We should preserve it.
#
# That template converts:
#
# [
#   system message,
#   user message,
#   assistant message
# ]
#
# into the exact special-token sequence the instruct model
# was trained to understand.
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)


print(
    "Vocabulary size:",
    len(tokenizer)
)

print(
    "EOS:",
    tokenizer.eos_token
)

print(
    "PAD:",
    tokenizer.pad_token
)

print(
    "Chat template available:",
    tokenizer.chat_template is not None
)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Vocabulary size: 49152
EOS: <|im_end|>
PAD: <|im_end|>
Chat template available: True


In [ ]:
# ============================================================
# INSPECT CHAT TEMPLATE
# ============================================================
#
# This is useful technically.
#
# Your dataframe isn't fed directly into the Transformer.
#
# The tokenizer converts:
#
# system → special role tokens
# user → special role tokens
# assistant → special role tokens
#
# before tokenization.
# ============================================================

example_messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": df.iloc[0]["text"]
    },
    {
        "role": "assistant",
        "content": df.iloc[0]["bullet_points"]
    }
]


formatted = tokenizer.apply_chat_template(
    example_messages,
    tokenize=False
)


print(formatted)

<|im_start|>system
You convert English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, and trivial details.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add information unsupported by the source.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".<|im_end|>
<|im_start|>user
Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #3

In [ ]:
# ============================================================
# PROMPT-COMPLETION FORMAT
# ============================================================
#
# We DON'T simply create:
#
#     one giant text string
#
# Instead:
#
# PROMPT:
#
#     SYSTEM instruction
#     +
#     USER document
#
#
# COMPLETION:
#
#     ASSISTANT bullets
#
#
# Why?
#
# Because our objective is:
#
# P(
#    correct bullets
#    |
#    instruction + document
# )
#
#
# We want training loss primarily on the response.
#
# We DON'T need to teach SmolLM to reproduce the original
# article/document.
# ============================================================

records = []


for _, row in df.iterrows():

    records.append(
        {
            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": row["text"]
                }
            ],

            "completion": [
                {
                    "role": "assistant",
                    "content": row["bullet_points"]
                }
            ]
        }
    )


dataset = Dataset.from_list(
    records
)


print(dataset)

print()

print(
    dataset[0]
)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 500
})

{'prompt': [{'role': 'system', 'content': 'You convert English text into concise bullet points containing all materially important information.\n\nFollow these rules:\n\n- Extract all important and independently useful points.\n- The number of bullets must depend entirely on the information in the text.\n- Never use a fixed number of bullets.\n- Use one bullet for each distinct important point.\n- Combine details that naturally belong together.\n- Remove repetition, filler, and trivial details.\n- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.\n- Do not add information unsupported by the source.\n- Keep every bullet concise while preserving the original meaning.\n- Return only bullet points.\n- Start every bullet with "- ".'}, {'role': 'user', 'content': 'Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explo

In [ ]:
# ============================================================
# INTERNAL TRAINING VALIDATION
# ============================================================
#
# 95% train
# 5% validation
#
#
# Validation tells us whether training is generalizing.
#
#
# IMPORTANT:
#
# This is NOT a replacement for your external 500-example
# benchmark.
#
# Ideally:
#
# TRAIN DATA
#      ↓
# SFT
#
# completely separate
#
# 500 EXAMPLES
#      ↓
# final benchmark
#
#
# Never train on the benchmark.
# ============================================================

dataset = dataset.train_test_split(
    test_size=0.05,
    seed=SEED
)


train_dataset = dataset["train"]

validation_dataset = dataset["test"]


print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

Train: 475
Validation: 25


In [ ]:
# ============================================================
# TOKEN LENGTH ANALYSIS
# ============================================================
#
# Before blindly truncating to 2048, measure the data.
#
#
# Transformer training cost grows strongly with sequence
# length because self-attention operates across tokens.
#
#
# If:
#
# 95% <= 2048
#
# then 2048 is an excellent starting point.
#
#
# If many examples are >2048:
#
# later we can test 4096.
# ============================================================

def get_token_length(example):

    messages = (
        example["prompt"]
        +
        example["completion"]
    )

    tokens = tokenizer.apply_chat_template(
        messages,
        tokenize=True
    )

    return len(tokens)


lengths = [
    get_token_length(
        train_dataset[i]
    )
    for i in tqdm(
        range(len(train_dataset))
    )
]


length_stats = pd.Series(
    lengths
)


print(
    length_stats.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


print()

print(
    "> 2048:",
    sum(
        x > 2048
        for x in lengths
    )
)

print(
    "> 4096:",
    sum(
        x > 4096
        for x in lengths
    )
)

  0%|          | 0/475 [00:00<?, ?it/s]

count    475.0
mean       2.0
std        0.0
min        2.0
50%        2.0
75%        2.0
90%        2.0
95%        2.0
99%        2.0
max        2.0
dtype: float64

> 2048: 0
> 4096: 0


In [ ]:
# ============================================================
# LOAD MODEL FOR FULL SFT
# ============================================================
#
# We load model weights in FP32 initially.
#
# During training:
#
#     fp16=True
#
# enables mixed-precision training on the T4.
#
#
# FULL FINE-TUNING means:
#
#     ALL model parameters receive gradients.
#
#
# This is different from LoRA:
#
# Full SFT:
#
#     135M / 135M parameters train
#
#
# LoRA:
#
#     base 135M frozen
#     only small adapter matrices train
#
#
# Because 135M is tiny, full SFT is realistic and gives us
# the strongest specialization experiment.
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
)


# KV cache is useful during generation.
#
# It is NOT needed during training and can conflict with
# gradient checkpointing.
model.config.use_cache = False


print(
    "Model loaded"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded


In [ ]:
# ============================================================
# VERIFY FULL FINE-TUNING
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Total:",
    f"{total_params:,}"
)

print(
    "Trainable:",
    f"{trainable_params:,}"
)

print(
    "Trainable %:",
    round(
        trainable_params
        / total_params
        * 100,
        2
    )
)

Total: 134,515,008
Trainable: 134,515,008
Trainable %: 100.0


## Custom evaluation

In [ ]:
# ============================================================
# FIXED GENERATION VALIDATION SET
# ============================================================
#
# IMPORTANT:
#
# This is different from the validation dataset used for
# eval_loss.
#
# We take a FIXED subset and actually generate answers.
#
#
# Why fixed?
#
# Epoch 1, 2, 3 ... must all see EXACTLY the same evaluation
# examples.
#
#
# Why small?
#
# Generation is much more expensive than teacher-forced
# evaluation.
#
# With 15 epochs:
#
# 20 examples × 15 epochs = 300 generations
#
# That's reasonable.
#
#
# Your final untouched benchmark should STILL be evaluated
# only after training.
# ============================================================

GEN_EVAL_SIZE = min(
    15,
    len(validation_dataset)
)


generation_eval_dataset = (
    validation_dataset.select(
        range(GEN_EVAL_SIZE)
    )
)


print(
    "Generation evaluation examples:",
    len(generation_eval_dataset)
)

Generation evaluation examples: 15


In [ ]:
# ============================================================
# ROUGE SCORER
# ============================================================

rouge_scorer_obj = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rouge2",
        "rougeL"
    ],
    use_stemmer=True
)

In [ ]:
# ============================================================
# TASK-SPECIFIC METRIC HELPERS
# ============================================================


def bullet_format_score(text):

    if not isinstance(text, str):
        return 0.0

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return 0.0

    correct = sum(
        line.startswith("- ")
        for line in lines
    )

    return correct / len(lines)



def count_bullets(text):

    if not isinstance(text, str):
        return 0

    return sum(
        line.strip().startswith("- ")
        for line in text.splitlines()
    )



def word_count(text):

    if not isinstance(text, str):
        return 0

    return len(
        text.split()
    )

In [ ]:
# ============================================================
# GENERATION-BASED EVALUATION
# ============================================================
#
# THIS is much closer to production behavior than eval_loss.
#
#
# For every held-out example:
#
# source paragraph
#       ↓
# model.generate()
#       ↓
# generated bullets
#       ↓
# compare against reference
#
#
# IMPORTANT:
#
# do_sample=False
#
# makes evaluation deterministic.
#
# We don't want random sampling to change scores between
# epochs.
#
# ============================================================


def run_generation_evaluation(
    model,
    tokenizer,
    dataset,
    max_new_tokens=512
):

    model.eval()

    device = next(
        model.parameters()
    ).device


    references = []
    predictions = []
    source_texts = []


    for example in tqdm(
        dataset,
        desc="Generation eval",
        leave=False
    ):

        # --------------------------------------------
        # Extract original paragraph
        # --------------------------------------------

        source_text = (
            example["prompt"][1]["content"]
        )


        # --------------------------------------------
        # Extract teacher/reference bullets
        # --------------------------------------------

        reference = (
            example["completion"][0]["content"]
        )


        # --------------------------------------------
        # Reconstruct inference prompt
        # --------------------------------------------

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": source_text
            }
        ]


        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )


        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH
        )


        inputs = {
            key: value.to(device)
            for key, value
            in inputs.items()
        }


        input_length = (
            inputs[
                "input_ids"
            ].shape[-1]
        )


        # --------------------------------------------
        # Actual autoregressive generation
        # --------------------------------------------

        with torch.inference_mode():

            outputs = model.generate(

                **inputs,

                max_new_tokens=max_new_tokens,

                do_sample=False,

                temperature=None,

                top_p=None,

                pad_token_id=(
                    tokenizer.eos_token_id
                )
            )


        generated_tokens = (
            outputs[
                0,
                input_length:
            ]
        )


        prediction = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()


        references.append(
            reference
        )

        predictions.append(
            prediction
        )

        source_texts.append(
            source_text
        )


    # ========================================================
    # ROUGE
    # ========================================================

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []


    for reference, prediction in zip(
        references,
        predictions
    ):

        scores = rouge_scorer_obj.score(
            reference,
            prediction
        )

        rouge1_scores.append(
            scores[
                "rouge1"
            ].fmeasure
        )

        rouge2_scores.append(
            scores[
                "rouge2"
            ].fmeasure
        )

        rougeL_scores.append(
            scores[
                "rougeL"
            ].fmeasure
        )


    # ========================================================
    # BERTSCORE
    # ========================================================
    #
    # Semantic comparison.
    #
    # Recall is particularly important for our task because:
    #
    #     Did we capture the important reference information?
    #
    # ========================================================

    P, R, F1 = bert_score(

        predictions,

        references,

        lang="en",

        device=str(device),

        verbose=False
    )


    # ========================================================
    # BULLET BEHAVIOR
    # ========================================================

    format_scores = [
        bullet_format_score(
            prediction
        )
        for prediction in predictions
    ]


    predicted_bullet_counts = [
        count_bullets(
            prediction
        )
        for prediction in predictions
    ]


    reference_bullet_counts = [
        count_bullets(
            reference
        )
        for reference in references
    ]


    # ========================================================
    # COMPRESSION
    # ========================================================

    compression_ratios = []


    for source, prediction in zip(
        source_texts,
        predictions
    ):

        source_words = max(
            word_count(source),
            1
        )

        prediction_words = (
            word_count(
                prediction
            )
        )

        compression_ratios.append(
            prediction_words
            /
            source_words
        )


    # ========================================================
    # RETURN AGGREGATED METRICS
    # ========================================================

    metrics = {

        "gen_rouge1":
            float(
                np.mean(
                    rouge1_scores
                )
            ),

        "gen_rouge2":
            float(
                np.mean(
                    rouge2_scores
                )
            ),

        "gen_rougeL":
            float(
                np.mean(
                    rougeL_scores
                )
            ),

        "gen_bertscore_precision":
            float(
                P.mean().item()
            ),

        "gen_bertscore_recall":
            float(
                R.mean().item()
            ),

        "gen_bertscore_f1":
            float(
                F1.mean().item()
            ),

        "gen_bullet_format":
            float(
                np.mean(
                    format_scores
                )
            ),

        "gen_predicted_bullets":
            float(
                np.mean(
                    predicted_bullet_counts
                )
            ),

        "gen_reference_bullets":
            float(
                np.mean(
                    reference_bullet_counts
                )
            ),

        "gen_compression_ratio":
            float(
                np.mean(
                    compression_ratios
                )
            )
    }


    return metrics

In [ ]:
# ============================================================
# CUSTOM TRAINER CALLBACK
# ============================================================
#
# Hugging Face already performs:
#
#     eval_loss
#
# at the end of every epoch because:
#
#     eval_strategy="epoch"
#
#
# When that evaluation finishes, this callback additionally
# performs REAL GENERATION and calculates our task metrics.
#
#
# So the training timeline becomes:
#
# Epoch 1
#    ↓
# teacher-forced validation
#    ↓
# eval_loss
# token accuracy
# entropy
#    ↓
# actual model.generate()
#    ↓
# ROUGE
# BERTScore
# bullet metrics
# compression
#    ↓
# save checkpoint
#
# Epoch 2
#    ↓
# same process
#
# ...
# ============================================================


class GenerationMetricsCallback(
    TrainerCallback
):

    def __init__(
        self,
        tokenizer,
        dataset
    ):

        self.tokenizer = tokenizer

        self.dataset = dataset

        self.history = []


    def on_evaluate(
        self,
        args,
        state,
        control,
        model=None,
        metrics=None,
        **kwargs
    ):

        print()
        print(
            "=" * 70
        )

        print(
            f"GENERATION EVALUATION "
            f"— Epoch {state.epoch:.2f}"
        )

        print(
            "=" * 70
        )


        generation_metrics = (
            run_generation_evaluation(

                model=model,

                tokenizer=self.tokenizer,

                dataset=self.dataset
            )
        )


        # Add epoch so we can later create a leaderboard.
        record = {
            "epoch":
                float(
                    state.epoch
                ),

            **generation_metrics
        }


        # Include normal validation loss in the same history.
        if metrics is not None:

            if "eval_loss" in metrics:

                record[
                    "eval_loss"
                ] = float(
                    metrics[
                        "eval_loss"
                    ]
                )


        self.history.append(
            record
        )


        print()


        for key, value in (
            generation_metrics.items()
        ):

            print(
                f"{key}: "
                f"{value:.4f}"
            )


        print(
            "=" * 70
        )

        print()


        # Return model to training mode.
        model.train()

In [ ]:
# ============================================================
# FULL SFT CONFIGURATION — SmolLM2-135M-Instruct
# ============================================================
#
# TWO TYPES OF EVALUATION NOW HAPPEN EVERY EPOCH:
#
#
# 1. TEACHER-FORCED EVALUATION
#
#       eval_loss
#       token accuracy
#       entropy
#
#
# 2. GENERATION EVALUATION
#
#       ROUGE-1
#       ROUGE-2
#       ROUGE-L
#       BERTScore P/R/F1
#       bullet format
#       bullet count
#       compression ratio
#
#
# Every epoch is also saved.
# ============================================================


training_args = SFTConfig(

    output_dir=OUTPUT_DIR,


    # --------------------------------------------------------
    # Sequence
    # --------------------------------------------------------

    max_length=MAX_LENGTH,

    completion_only_loss=True,


    # --------------------------------------------------------
    # 7 epochs
    # --------------------------------------------------------

    num_train_epochs=7,


    # --------------------------------------------------------
    # Batch
    # --------------------------------------------------------

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,


    # --------------------------------------------------------
    # Optimization
    # --------------------------------------------------------

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_steps=50,

    lr_scheduler_type="cosine",


    # --------------------------------------------------------
    # T4
    # --------------------------------------------------------

    fp16=True,

    bf16=False,


    gradient_checkpointing=True,


    # --------------------------------------------------------
    # EVALUATE EVERY EPOCH
    # --------------------------------------------------------

    eval_strategy="epoch",


    # --------------------------------------------------------
    # SAVE EVERY EPOCH
    # --------------------------------------------------------

    save_strategy="epoch",

    save_total_limit=None,


    # --------------------------------------------------------
    # Restore checkpoint with lowest validation loss
    #
    # NOTE:
    # We'll separately inspect generation metrics afterward.
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,


    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_steps=10,


    seed=SEED,

    report_to="none"
)

In [ ]:
# ============================================================
# CREATE GENERATION CALLBACK
# ============================================================

generation_callback = (
    GenerationMetricsCallback(

        tokenizer=tokenizer,

        dataset=(
            generation_eval_dataset
        )
    )
)

In [ ]:
# ============================================================
# CREATE TRAINER
# ============================================================

trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    callbacks=[
        generation_callback
    ]
)


print(
    "Trainer ready"
)

Tokenizing train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/475 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Trainer ready


In [ ]:
# ============================================================
# TRAIN
# ============================================================

start_time = time.time()


train_result = trainer.train()


training_time = (
    time.time()
    -
    start_time
)


print()

print(
    "Training time:",
    round(
        training_time / 60,
        2
    ),
    "minutes"
)

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.230977,0.183213,0.137878,320131.000000,0.964828
2,0.121642,0.115885,0.106419,640262.000000,0.977893
3,0.177864,0.095775,0.085583,960393.000000,0.980868
4,0.051406,0.091187,0.071412,1280524.000000,0.980891
5,0.040507,0.090018,0.068408,1600655.000000,0.980534
6,0.034172,0.089348,0.064705,1920786.000000,0.980657
7,0.030540,0.089770,0.063962,2240917.000000,0.981159



GENERATION EVALUATION — Epoch 1.00


Generation eval:   0%|          | 0/15 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



gen_rouge1: 0.8957
gen_rouge2: 0.8769
gen_rougeL: 0.8812
gen_bertscore_precision: 0.9726
gen_bertscore_recall: 0.9782
gen_bertscore_f1: 0.9753
gen_bullet_format: 1.0000
gen_predicted_bullets: 4.8000
gen_reference_bullets: 4.7333
gen_compression_ratio: 0.7490



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Training time: 29.17 minutes


In [ ]:
# ============================================================
# FINAL TRAINING METRICS
# ============================================================

train_result.metrics

{'train_runtime': 1743.6721,
 'train_samples_per_second': 1.875,
 'train_steps_per_second': 0.12,
 'total_flos': 2536283815607808.0,
 'train_loss': 0.10955395982379006,
 'epoch': 7.0}

In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

validation_metrics = trainer.evaluate()

validation_metrics


GENERATION EVALUATION — Epoch 7.00


Generation eval:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



gen_rouge1: 0.8957
gen_rouge2: 0.8769
gen_rougeL: 0.8812
gen_bertscore_precision: 0.9724
gen_bertscore_recall: 0.9780
gen_bertscore_f1: 0.9751
gen_bullet_format: 1.0000
gen_predicted_bullets: 4.7333
gen_reference_bullets: 4.7333
gen_compression_ratio: 0.7489



Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.030540,0.089348,7,0.064705,2240917.000000,0.980657


{'eval_loss': 0.08934842795133591,
 'eval_entropy': 0.06470488544021334,
 'eval_num_tokens': 2240917.0,
 'eval_mean_token_accuracy': 0.9806569048336574}

In [ ]:
# ============================================================
# SHOW ALL SAVED EPOCH CHECKPOINTS
# ============================================================

import glob
import os


checkpoints = sorted(
    glob.glob(
        os.path.join(
            OUTPUT_DIR,
            "checkpoint-*"
        )
    ),
    key=lambda path: int(
        path.split("-")[-1]
    )
)


print(
    "Total checkpoints:",
    len(checkpoints)
)

print()


for i, checkpoint in enumerate(
    checkpoints,
    start=1
):

    print(
        f"Epoch {i:02d}: {checkpoint}"
    )

Total checkpoints: 7

Epoch 01: /content/smollm2-135m-bullet-sft/checkpoint-30
Epoch 02: /content/smollm2-135m-bullet-sft/checkpoint-60
Epoch 03: /content/smollm2-135m-bullet-sft/checkpoint-90
Epoch 04: /content/smollm2-135m-bullet-sft/checkpoint-120
Epoch 05: /content/smollm2-135m-bullet-sft/checkpoint-150
Epoch 06: /content/smollm2-135m-bullet-sft/checkpoint-180
Epoch 07: /content/smollm2-135m-bullet-sft/checkpoint-210


In [ ]:
# ============================================================
# BEST CHECKPOINT SELECTED BY VALIDATION LOSS
# ============================================================
#
# Trainer tracks:
#
#     best_model_checkpoint
#     best_metric
#
# ============================================================

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation loss:",
    trainer.state.best_metric
)

Best checkpoint: /content/smollm2-135m-bullet-sft/checkpoint-180
Best validation loss: 0.08934842795133591


In [ ]:
# ============================================================
# SAVE BEST MODEL AS FINAL SPECIALIST
# ============================================================
#
# trainer.model currently contains the BEST checkpoint
# according to validation loss.
#
# We save it separately so deployment code doesn't need to
# know about checkpoint-XX.
#
#
# Original:
#
# HuggingFaceTB/SmolLM2-135M-Instruct
#
#            ↓ FULL SFT
#
# Our specialist:
#
# smollm2-135m-keypoints
# ============================================================

FINAL_MODEL_DIR = (
    "/content/smollm2-135m-keypoints"
)


trainer.save_model(
    FINAL_MODEL_DIR
)


tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


print(
    "Best model saved to:",
    FINAL_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /content/smollm2-135m-keypoints


In [ ]:
# ============================================================
# VERIFY FINAL MODEL
# ============================================================

!du -sh /content/smollm2-135m-keypoints

!ls -lh /content/smollm2-135m-keypoints

517M	/content/smollm2-135m-keypoints
total 517M
-rw-r--r-- 1 root root  368 Sep  5 06:59 chat_template.jinja
-rw-r--r-- 1 root root  906 Sep  5 06:59 config.json
-rw-r--r-- 1 root root  142 Sep  5 06:59 generation_config.json
-rw------- 1 root root 514M Sep  5 06:59 model.safetensors
-rw-r--r-- 1 root root  453 Sep  5 06:59 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Sep  5 06:59 tokenizer.json
-rw-r--r-- 1 root root 5.7K Sep  5 06:59 training_args.bin


In [ ]:
# ============================================================
# HUGGING FACE LOGIN
# ============================================================
#
# In Google Colab:
#
# 1. Open Secrets (key icon)
# 2. Add:
#
#       HF_TOKEN
#
# 3. Paste your Hugging Face WRITE token
#
# 4. Enable notebook access
#
#
# Never hard-code the token into a notebook that may be
# shared.
# ============================================================

from google.colab import userdata

from huggingface_hub import login


HF_TOKEN = ""


login(
    token=HF_TOKEN
)


print(
    "Logged into Hugging Face"
)

Logged into Hugging Face


In [ ]:
# ============================================================
# CREATE HUGGING FACE MODEL REPOSITORY
# ============================================================
#
# exist_ok=True:
#
# rerunning this cell won't fail if the repository already
# exists.
#
#
# private=False:
#
# public model.
#
# Change to:
#
# private=True
#
# if you don't want it publicly visible yet.
# ============================================================

from huggingface_hub import create_repo

HF_REPO_ID = "JayShah07/smol-text-bullet-135m"

create_repo(

    repo_id=HF_REPO_ID,

    repo_type="model",

    private=False,

    exist_ok=True
)


print(
    "Repository ready:",
    HF_REPO_ID
)

Repository ready: JayShah07/smol-text-bullet-135m


In [ ]:
# ============================================================
# UPLOAD BEST MODEL
# ============================================================
#
# We are intentionally uploading the clean BEST model,
# NOT all 15 training checkpoints.
#
#
# Hugging Face repo will therefore contain something like:
#
# config.json
# generation_config.json
# model.safetensors
# tokenizer.json
# tokenizer_config.json
# ...
#
#
# Users can later simply do:
#
# AutoModelForCausalLM.from_pretrained(HF_REPO_ID)
#
# ============================================================

from huggingface_hub import upload_folder


upload_folder(

    repo_id=HF_REPO_ID,

    repo_type="model",

    folder_path=FINAL_MODEL_DIR,

    commit_message=(
        "Upload best SmolLM2-135M bullet specialist"
    )
)


print(
    "Uploaded successfully:"
)

print(
    f"https://huggingface.co/{HF_REPO_ID}"
)

Uploaded successfully:
https://huggingface.co/JayShah07/smol-text-bullet-135m


In [ ]:
# ============================================================
# FREE TRAINING MEMORY
# ============================================================

del trainer
del model

gc.collect()

torch.cuda.empty_cache()

print(
    "Training memory cleared"
)

Training memory cleared


## Evaluation

In [ ]:
# ============================================================
# DOWNLOAD OUR FINE-TUNED MODEL
# ============================================================
#
# from_pretrained() automatically:
#
# 1. connects to Hugging Face
# 2. downloads config.json
# 3. downloads model.safetensors
# 4. downloads tokenizer files
# 5. caches them in the Colab environment
# 6. constructs the model
#
# We use FP16 on the T4 for inference.
# ============================================================

MODEL_ID = "JayShah07/smol-text-bullet-135m"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16
)


model = model.to("cuda")

model.eval()


print("Loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/906 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/368 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Loaded: JayShah07/smol-text-bullet-135m
Device: cuda:0


In [ ]:
# ============================================================
# LOAD SPECIALIST IN FP16 FOR INFERENCE
# ============================================================
#
# Training and inference are separate concerns.
#
# During inference we don't need gradients.
#
# FP16 cuts model memory roughly in half relative to FP32.
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    FINAL_MODEL_DIR,
    dtype=torch.float16
)


model = model.to("cuda")

model.eval()


tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)


print(
    "Specialist loaded"
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Specialist loaded


In [ ]:
# ============================================================
# LOAD DOWNLOADED SPECIALIST IN FP16 FOR INFERENCE
# ============================================================
#
# The fine-tuned model is now stored on Hugging Face:
#
#     JayShah07/smol-text-bullet-135m
#
# from_pretrained() will automatically download/cache:
#
#     model weights
#     config
#     tokenizer
#     chat template
#     generation config
#
# FP16 is used for efficient inference on the T4 GPU.
#
# ============================================================

MODEL_ID = "JayShah07/smol-text-bullet-135m"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16
)


model = model.to("cuda")

model.eval()


print(
    "Specialist loaded"
)

print(
    "Model:",
    MODEL_ID
)

print(
    "Device:",
    next(model.parameters()).device
)

print(
    "Dtype:",
    next(model.parameters()).dtype
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Specialist loaded
Model: JayShah07/smol-text-bullet-135m
Device: cuda:0
Dtype: torch.float16


In [ ]:
# ============================================================
# GENERATION FUNCTION
# ============================================================

MAX_LENGTH = 2048


def generate_bullets(
    text,
    max_new_tokens=512
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": text
        }
    ]


    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )


    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }


    input_length = (
        inputs["input_ids"].shape[-1]
    )


    torch.cuda.synchronize()

    start = time.perf_counter()


    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )


    torch.cuda.synchronize()

    latency = (
        time.perf_counter()
        - start
    )


    generated_tokens = outputs[
        0,
        input_length:
    ]


    output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


    return {
        "output": output,
        "latency_seconds": latency,
        "output_tokens": len(generated_tokens)
    }

In [ ]:
# ============================================================
# UNSEEN MANUAL TEST
# ============================================================

test_text = """
Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
Operating profit increased 8% to $620 million, although operating margin
declined from 17.2% to 14.8%. The company added 1.3 million customers
during the quarter and raised full-year revenue guidance from $16 billion
to $17.5 billion. Management warned that European demand weakened in July.
"""


result = generate_bullets(
    test_text
)


print(
    result["output"]
)

print()

print(
    "Latency:",
    round(
        result["latency_seconds"],
        3
    ),
    "seconds"
)

print(
    "Generated tokens:",
    result["output_tokens"]
)

- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 5.192 seconds
Generated tokens: 101


In [ ]:
# ============================================================
# LOAD EXTERNAL TEST SET
# ============================================================
#
# IMPORTANT:
#
# This must NOT be data that was used during SFT.
#
# This is our real evaluation set.
#
# Expected columns:
#
#   text
#   source
#   example_id
#   bullet_points
#
# bullet_points = reference/teacher answer
#
#
# We will compare:
#
#     REFERENCE
#        vs
#     MODEL GENERATION
#
# ============================================================

TEST_CSV = "/content/synthetic_evaluation_500.csv"

test_df = pd.read_csv(TEST_CSV)

print("Test examples:", len(test_df))
print("Columns:", test_df.columns.tolist())

test_df.head()

Test examples: 500
Columns: ['text', 'source', 'example_id', 'bullet_points']


,text,source,example_id,bullet_points
0,Riverton (Synthetic News) -- Officials in Dist...,cnn_dailymail,synthetic_cnn_dailymail_001,- Riverton officials reviewed a plan on a dela...
1,Lakeside (Synthetic News) -- Officials in Dist...,cnn_dailymail,synthetic_cnn_dailymail_002,- Lakeside officials reviewed a plan on hospit...
2,Marina City (Synthetic News) -- Officials in D...,cnn_dailymail,synthetic_cnn_dailymail_003,- Marina City officials reviewed a plan on ten...
3,Hillford (Synthetic News) -- Officials in Dist...,cnn_dailymail,synthetic_cnn_dailymail_004,- Hillford officials reviewed a plan on a dela...
4,Brookhaven (Synthetic News) -- Officials in Di...,cnn_dailymail,synthetic_cnn_dailymail_005,- Brookhaven officials reviewed a plan on hosp...


In [ ]:
# ============================================================
# GENERATE ON HELD-OUT TEST DATA
# ============================================================
#
# For every example:
#
# source text
#      ↓
# fine-tuned 135M model
#      ↓
# generated bullets
#
# We retain the teacher/reference bullets separately.
#
# ============================================================

predictions = []


for value, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating SFT model"
):

    result = generate_bullets(
        row["text"]
    )


    predictions.append({
        "example_id":
            row["example_id"],

        "source":
            row["source"],

        "text":
            row["text"],

        "reference":
            row["bullet_points"],

        "prediction":
            result["output"],

        "latency_seconds":
            result["latency_seconds"],

        "output_tokens":
            result["output_tokens"]
    })


results_df = pd.DataFrame(
    predictions
)


print(
    "Generated:",
    len(results_df)
)

results_df.head()

Evaluating SFT model:   0%|          | 0/500 [00:00<?, ?it/s]

Generated: 500


,example_id,source,text,reference,prediction,latency_seconds,output_tokens
0,synthetic_cnn_dailymail_001,cnn_dailymail,Riverton (Synthetic News) -- Officials in Dist...,- Riverton officials reviewed a plan on a dela...,- Riverton (Synthetic News) -- Officials in Di...,5.557070,158
1,synthetic_cnn_dailymail_002,cnn_dailymail,Lakeside (Synthetic News) -- Officials in Dist...,- Lakeside officials reviewed a plan on hospit...,- Lakeside (Synthetic News) -- Officials in Di...,5.415887,155
2,synthetic_cnn_dailymail_003,cnn_dailymail,Marina City (Synthetic News) -- Officials in D...,- Marina City officials reviewed a plan on ten...,- Marina City (Synthetic News) -- Officials in...,5.397707,143
3,synthetic_cnn_dailymail_004,cnn_dailymail,Hillford (Synthetic News) -- Officials in Dist...,- Hillford officials reviewed a plan on a dela...,- Hillford (Synthetic News) -- Officials in Di...,5.421794,162
4,synthetic_cnn_dailymail_005,cnn_dailymail,Brookhaven (Synthetic News) -- Officials in Di...,- Brookhaven officials reviewed a plan on hosp...,- Brookhaven (Synthetic News) -- Officials in ...,5.361082,138


In [ ]:
# ============================================================
# SAVE RAW MODEL OUTPUTS
# ============================================================
#
# Keep raw generations permanently.
#
# This allows us to add new evaluation metrics later WITHOUT
# rerunning model inference.
# ============================================================

PREDICTIONS_FILE = (
    "/content/smollm135_sft_predictions.csv"
)

results_df.to_csv(
    PREDICTIONS_FILE,
    index=False
)

print(
    "Saved:",
    PREDICTIONS_FILE
)

Saved: /content/smollm135_sft_predictions.csv


In [ ]:
# ============================================================
# EVALUATION LIBRARIES
# ============================================================
#
# rouge-score:
#   lexical overlap
#
# bert-score:
#   semantic similarity using contextual embeddings
#
# sacrebleu:
#   BLEU metric
#
# We will NOT treat any single metric as "truth".
#
# Summarization/extraction quality has multiple dimensions.
# ============================================================

!pip install -q \
    rouge-score \
    bert-score \
    sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00


In [ ]:
from rouge_score import rouge_scorer

from bert_score import score as bert_score

import sacrebleu

In [ ]:
# ============================================================
# ROUGE
# ============================================================
#
# ROUGE-1
# --------
#
# Unigram/word overlap.
#
# Measures roughly:
#
# "Did the model preserve the important words/concepts?"
#
#
# ROUGE-2
# --------
#
# Bigram overlap.
#
# More sensitive to matching actual phrases/facts.
#
#
# ROUGE-L
# --------
#
# Longest-common-subsequence based.
#
# Captures similarity in content/order while being less
# strict than exact matching.
#
#
# For our experiment:
#
# ROUGE-L + ROUGE-2 are particularly useful.
# ============================================================

rouge_scorer_obj = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rouge2",
        "rougeL"
    ],
    use_stemmer=True
)


def calculate_rouge(
    reference,
    prediction
):

    scores = rouge_scorer_obj.score(
        reference,
        prediction
    )

    return {
        "rouge1":
            scores["rouge1"].fmeasure,

        "rouge2":
            scores["rouge2"].fmeasure,

        "rougeL":
            scores["rougeL"].fmeasure
    }

In [ ]:
rouge_results = [
    calculate_rouge(
        row["reference"],
        row["prediction"]
    )
    for _, row in tqdm(
        results_df.iterrows(),
        total=len(results_df)
    )
]


results_df["rouge1"] = [
    x["rouge1"]
    for x in rouge_results
]

results_df["rouge2"] = [
    x["rouge2"]
    for x in rouge_results
]

results_df["rougeL"] = [
    x["rougeL"]
    for x in rouge_results
]

  0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# BERTSCORE
# ============================================================
#
# Instead of comparing exact words, BERTScore compares
# contextual representations.
#
# Useful because generative models can express the SAME
# important point using different wording.
#
#
# Precision:
#
#   How much of what the model generated is supported/relevant?
#
#
# Recall:
#
#   How much of the reference information did it capture?
#
#
# F1:
#
#   Balance between the two.
#
#
# For our task BERTScore RECALL is especially interesting:
#
#   "Did we capture the important information?"
# ============================================================

P, R, F1 = bert_score(
    results_df["prediction"].tolist(),
    results_df["reference"].tolist(),

    lang="en",

    device="cuda",

    verbose=True
)


results_df[
    "bertscore_precision"
] = P.cpu().numpy()


results_df[
    "bertscore_recall"
] = R.cpu().numpy()


results_df[
    "bertscore_f1"
] = F1.cpu().numpy()

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 11.57 seconds, 43.21 sentences/sec


In [ ]:
# ============================================================
# BULLET FORMAT COMPLIANCE
# ============================================================
#
# Desired:
#
# - point
# - point
# - point
#
#
# Score = percentage of non-empty output lines beginning "- "
#
#
# 1.0 = perfect
# 0.0 = completely ignored format
# ============================================================

def bullet_format_score(text):

    if not isinstance(text, str):
        return 0.0


    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


    if not lines:
        return 0.0


    correct = sum(
        line.startswith("- ")
        for line in lines
    )


    return (
        correct / len(lines)
    )


results_df[
    "bullet_format_score"
] = (
    results_df[
        "prediction"
    ].apply(
        bullet_format_score
    )
)

In [ ]:
# ============================================================
# BULLET COUNT
# ============================================================
#
# We want variable bullet counts.
#
# Dense documents should naturally produce more bullets.
#
# Simple documents should produce fewer.
#
#
# If every output suddenly has exactly 5 bullets:
#
# something is wrong.
# ============================================================

def count_bullets(text):

    if not isinstance(text, str):
        return 0

    return sum(
        1
        for line in text.splitlines()
        if line.strip().startswith("- ")
    )


results_df[
    "prediction_bullets"
] = (
    results_df[
        "prediction"
    ].apply(
        count_bullets
    )
)


results_df[
    "reference_bullets"
] = (
    results_df[
        "reference"
    ].apply(
        count_bullets
    )
)

In [ ]:
# ============================================================
# COMPRESSION RATIO
# ============================================================
#
# prediction words / input words
#
#
# Example:
#
# input       = 500 words
# output      = 100 words
#
# ratio       = 0.20
#
#
# Lower isn't automatically better.
#
# Too low:
#   important information may be missing.
#
# Too high:
#   model isn't summarizing enough.
#
# ============================================================

def word_count(text):

    if not isinstance(text, str):
        return 0

    return len(
        text.split()
    )


results_df[
    "input_words"
] = (
    results_df[
        "text"
    ].apply(
        word_count
    )
)


results_df[
    "prediction_words"
] = (
    results_df[
        "prediction"
    ].apply(
        word_count
    )
)


results_df[
    "reference_words"
] = (
    results_df[
        "reference"
    ].apply(
        word_count
    )
)


results_df[
    "compression_ratio"
] = (
    results_df["prediction_words"]
    /
    results_df["input_words"]
)

In [ ]:
# ============================================================
# FINAL AUTOMATIC METRICS
# ============================================================

summary = pd.DataFrame([
    {
        "model":
            "SmolLM2-135M-SFT",

        "examples":
            len(results_df),

        "rouge1":
            results_df[
                "rouge1"
            ].mean(),

        "rouge2":
            results_df[
                "rouge2"
            ].mean(),

        "rougeL":
            results_df[
                "rougeL"
            ].mean(),

        "bertscore_precision":
            results_df[
                "bertscore_precision"
            ].mean(),

        "bertscore_recall":
            results_df[
                "bertscore_recall"
            ].mean(),

        "bertscore_f1":
            results_df[
                "bertscore_f1"
            ].mean(),

        "bullet_format":
            results_df[
                "bullet_format_score"
            ].mean(),

        "avg_predicted_bullets":
            results_df[
                "prediction_bullets"
            ].mean(),

        "avg_reference_bullets":
            results_df[
                "reference_bullets"
            ].mean(),

        "compression_ratio":
            results_df[
                "compression_ratio"
            ].mean(),

        "avg_latency_seconds":
            results_df[
                "latency_seconds"
            ].mean()
    }
])


summary.T

,0
model,SmolLM2-135M-SFT
examples,500
rouge1,0.431802
rouge2,0.237415
rougeL,0.308292
bertscore_precision,0.876947
bertscore_recall,0.909809
bertscore_f1,0.893007
bullet_format,1.0
avg_predicted_bullets,3.428


In [ ]:
# ============================================================
# ZERO-SHOT VS SFT
# ============================================================
#
# These zero-shot numbers come from your earlier 500-example
# experiment.
#
# The question:
#
#       How much capability did specialization buy us?
# ============================================================

baseline = {
    "rouge1": 0.452839,
    "rouge2": 0.317955,
    "rougeL": 0.372168,
    "bullet_format": 0.082755
}


comparison = pd.DataFrame([
    {
        "model":
            "SmolLM2-135M Zero-shot",

        "rouge1":
            baseline["rouge1"],

        "rouge2":
            baseline["rouge2"],

        "rougeL":
            baseline["rougeL"],

        "bullet_format":
            baseline["bullet_format"]
    },

    {
        "model":
            "SmolLM2-135M SFT",

        "rouge1":
            results_df["rouge1"].mean(),

        "rouge2":
            results_df["rouge2"].mean(),

        "rougeL":
            results_df["rougeL"].mean(),

        "bullet_format":
            results_df[
                "bullet_format_score"
            ].mean()
    }
])


comparison

,model,rouge1,rouge2,rougeL,bullet_format
0,SmolLM2-135M Zero-shot,0.452839,0.317955,0.372168,0.082755
1,SmolLM2-135M SFT,0.431802,0.237415,0.308292,1.000000


In [ ]:
# ============================================================
# ABSOLUTE IMPROVEMENT FROM SPECIALIZATION
# ============================================================

print(
    "ROUGE-1 improvement:",
    round(
        results_df["rouge1"].mean()
        - baseline["rouge1"],
        4
    )
)


print(
    "ROUGE-2 improvement:",
    round(
        results_df["rouge2"].mean()
        - baseline["rouge2"],
        4
    )
)


print(
    "ROUGE-L improvement:",
    round(
        results_df["rougeL"].mean()
        - baseline["rougeL"],
        4
    )
)


print(
    "Bullet-format improvement:",
    round(
        results_df[
            "bullet_format_score"
        ].mean()
        - baseline["bullet_format"],
        4
    )
)

ROUGE-1 improvement: -0.021
ROUGE-2 improvement: -0.0805
ROUGE-L improvement: -0.0639
Bullet-format improvement: 0.9172


In [ ]:
# ============================================================
# CAN 135M SPECIALIZATION APPROACH 600M GENERAL MODEL?
# ============================================================

qwen_baseline = {
    "rouge1": 0.649384,
    "rouge2": 0.470194,
    "rougeL": 0.546342,
    "bullet_format": 0.999923
}


size_comparison = pd.DataFrame([
    {
        "model":
            "SmolLM2-135M SFT",

        "parameters_m":
            135,

        "rouge1":
            results_df[
                "rouge1"
            ].mean(),

        "rouge2":
            results_df[
                "rouge2"
            ].mean(),

        "rougeL":
            results_df[
                "rougeL"
            ].mean(),

        "bullet_format":
            results_df[
                "bullet_format_score"
            ].mean()
    },

    {
        "model":
            "Qwen3-0.6B Zero-shot",

        "parameters_m":
            600,

        "rouge1":
            qwen_baseline[
                "rouge1"
            ],

        "rouge2":
            qwen_baseline[
                "rouge2"
            ],

        "rougeL":
            qwen_baseline[
                "rougeL"
            ],

        "bullet_format":
            qwen_baseline[
                "bullet_format"
            ]
    }
])


size_comparison

,model,parameters_m,rouge1,rouge2,rougeL,bullet_format
0,SmolLM2-135M SFT,135,0.431802,0.237415,0.308292,1.000000
1,Qwen3-0.6B Zero-shot,600,0.649384,0.470194,0.546342,0.999923


In [ ]:
# ============================================================
# DOMAIN ANALYSIS
# ============================================================

domain_summary = (
    results_df
    .groupby(
        "source"
    )
    .agg(
        examples=(
            "example_id",
            "count"
        ),

        rouge1=(
            "rouge1",
            "mean"
        ),

        rouge2=(
            "rouge2",
            "mean"
        ),

        rougeL=(
            "rougeL",
            "mean"
        ),

        bertscore_f1=(
            "bertscore_f1",
            "mean"
        ),

        bullet_format=(
            "bullet_format_score",
            "mean"
        ),

        compression_ratio=(
            "compression_ratio",
            "mean"
        ),

        latency=(
            "latency_seconds",
            "mean"
        )
    )
    .reset_index()
)


domain_summary

,source,examples,rouge1,rouge2,rougeL,bertscore_f1,bullet_format,compression_ratio,latency
0,ag_news,125,0.391109,0.176273,0.247133,0.885757,1.0,1.062053,2.415708
1,cnn_dailymail,125,0.491703,0.238972,0.354603,0.898162,1.0,0.751972,5.258429
2,xsum,125,0.428513,0.255883,0.282423,0.905482,1.0,1.052704,2.650929
3,yahoo_answers,125,0.415884,0.278533,0.349010,0.882628,1.0,0.851113,2.217549


In [ ]:
results_df[
    [
        "example_id",
        "source",
        "rougeL",
        "bertscore_f1"
    ]
].sort_values(
    "rougeL",
    ascending=False
).head(10)

,example_id,source,rougeL,bertscore_f1
6,6,email,0.863636,0.964760
10,10,cnn_dailymail,0.743243,0.954202
3,3,xsum,0.704545,0.930300
2,2,ag_news,0.688525,0.924644
7,7,cnn_dailymail,0.622061,0.935431
9,9,email,0.522613,0.912481
5,5,email,0.513889,0.915549
0,0,ag_news,0.476190,0.906897
4,4,cnn_dailymail,0.255172,0.862786
1,1,cnn_dailymail,0.113164,0.747945


In [ ]:
worst = (
    results_df
    .sort_values(
        "rougeL",
        ascending=True
    )
    .head(10)
)


for _, row in worst.iterrows():

    print(
        "=" * 100
    )

    print(
        "EXAMPLE:",
        row["example_id"]
    )

    print(
        "SOURCE:",
        row["source"]
    )

    print(
        "ROUGE-L:",
        round(
            row["rougeL"],
            3
        )
    )

    print(
        "\nINPUT:\n"
    )

    print(
        row["text"]
    )

    print(
        "\nREFERENCE:\n"
    )

    print(
        row["reference"]
    )

    print(
        "\nMODEL:\n"
    )

    print(
        row["prediction"]
    )

    print()

EXAMPLE: synthetic_yahoo_answers_046
SOURCE: yahoo_answers
ROUGE-L: 0.063

INPUT:

Question: How do I troubleshoot community college transfer credits without spending much money as a student? Additional details: I have found several conflicting explanations online and want a practical answer that is easy to check. Best answer: Look for recent explanations that show their assumptions instead of relying on short forum replies. Reference note: synthetic question case 2045.

REFERENCE:

- Topic: Sports.
- User asks about community college transfer credits without spending much money as a student.
- Best-answer guidance emphasizes verification, practical comparison, and checking reliable sources.
- Synthetic question case: 2045.

MODEL:

- Check for recent explanations that show their assumptions instead of relying on short forum replies.
- Best answer: Look for recent explanations that show their assumptions instead of relying on short forum replies.

EXAMPLE: synthetic_yahoo_answers_086
S